In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Phase 1 - Target & Leakage Boundary

In [7]:
RANDOM_STATE = 123
DATA_DIR = '../02_data_cleaning/data/clean'

In [8]:
orders = pd.read_csv(f"{DATA_DIR}/Olist_Orders.csv",
                      parse_dates=["order_purchase_timestamp", "order_approved_at",
                                   "order_delivered_carrier_date",
                                   "order_delivered_customer_date",
                                   "order_estimated_delivery_date"])
items = pd.read_csv(f"{DATA_DIR}/Olist_Order_Items.csv")
customers = pd.read_csv(f"{DATA_DIR}/Olist_Customers.csv")
sellers = pd.read_csv(f"{DATA_DIR}/Olist_Sellers.csv")
products = pd.read_csv(f"{DATA_DIR}/Olist_Products.csv")
payments = pd.read_csv(f"{DATA_DIR}/Olist_Order_Payments.csv")

orders = orders[orders["order_status"] == "delivered"].copy()

In [11]:
def clean_unnamed(df):
    return df.loc[:, ~df.columns.str.contains("^Unnamed")]

orders = clean_unnamed(orders)
items = clean_unnamed(items)
customers = clean_unnamed(customers)
sellers = clean_unnamed(sellers)
products = clean_unnamed(products)
payments = clean_unnamed(payments)

In [12]:
df = (orders
      .merge(items, on="order_id", how="left")
      .merge(customers, on="customer_id", how="left")
      .merge(sellers, on="seller_id", how="left")
      .merge(products, on="product_id", how="left")
      .merge(payments, on="order_id", how="left"))

In [13]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,was_delivered,order_item_id,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,1,...,268.0,4.0,500.0,19.0,8.0,13.0,1.0,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,1,...,268.0,4.0,500.0,19.0,8.0,13.0,3.0,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,1,...,268.0,4.0,500.0,19.0,8.0,13.0,2.0,voucher,1.0,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,1,...,178.0,1.0,400.0,19.0,13.0,19.0,1.0,boleto,1.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,1,...,232.0,1.0,420.0,24.0,19.0,21.0,1.0,credit_card,3.0,179.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115033,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,True,1,...,828.0,4.0,4950.0,40.0,10.0,40.0,1.0,credit_card,3.0,195.00
115034,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,True,1,...,500.0,2.0,13300.0,32.0,90.0,22.0,1.0,credit_card,5.0,271.01
115035,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,True,1,...,1893.0,1.0,6550.0,20.0,20.0,20.0,1.0,credit_card,4.0,441.16
115036,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,True,2,...,1893.0,1.0,6550.0,20.0,20.0,20.0,1.0,credit_card,4.0,441.16


In [14]:
df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)

In [19]:
raw_col_count = df.shape[1]
raw_col_count

35

In [16]:
print(df.shape, df['is_late'].value_counts(normalize=True))

(115038, 35) is_late
0    0.921539
1    0.078461
Name: proportion, dtype: float64


# Phase 2 - Time-Based Split

In [20]:
df = df.sort_values('order_purchase_timestamp')
idx = int(len(df) * 0.8)
date = df.iloc[idx]['order_purchase_timestamp']

In [21]:
train = df[df['order_purchase_timestamp'] < date].copy()
test = df[df['order_purchase_timestamp'] >= date].copy()

In [22]:
print(train.shape, test.shape)

(92030, 35) (23008, 35)


# Phase 3 - Feature Creation

In [23]:
for split in (train, test):
    ts = split['order_purchase_timestamp']
    split['purchase_weekday'] = ts.dt.dayofweek
    split['purchase_month'] = ts.dt.month
    split['purchase_hour'] = ts.dt.hour
    split['is_weekend'] = split['purchase_weekday'].isin([5, 6]).astype(int)

    split["freight_to_price_ratio"] = split["freight_value"] / split["price"].replace(0, np.nan)
    split["item_count"] = split.groupby("order_id")["order_item_id"].transform("count")
    split["product_weight_g"] = split["product_weight_g"].fillna(split["product_weight_g"].median())
    split["product_volume_cm3"] = (split["product_length_cm"].fillna(0) * split["product_height_cm"].fillna(0) * split["product_width_cm"].fillna(0))

    split["same_state"] = (split["seller_state"] == split["customer_state"]).astype(int)

In [24]:
seller_late_rate = train.groupby("seller_id")["is_late"].mean()
global_late_rate = train["is_late"].mean()
category_avg_freight = train.groupby("product_category_name")["freight_value"].mean()
global_avg_freight = train["freight_value"].mean()

In [25]:
for split in (train, test):
    split["seller_late_rate_hist"] = split["seller_id"].map(seller_late_rate).fillna(global_late_rate)
    split["category_avg_freight_hist"] = (split["product_category_name"].map(category_avg_freight).fillna(global_avg_freight))

In [28]:
created_feature_count = len(["purchase_weekday","purchase_month","purchase_hour","is_weekend",
                              "freight_to_price_ratio","item_count","product_weight_g",
                              "product_volume_cm3","same_state","seller_late_rate_hist",
                              "category_avg_freight_hist"])

print(f'We have created {created_feature_count} new feature')

We have created 11 new feature


# Phase 4 - Encoding

In [29]:
high_card_cols = ["product_category_name", "seller_state", "customer_state"]

In [33]:
encoder = TargetEncoder(target_type="binary", smooth=10, random_state=RANDOM_STATE)
train[high_card_cols] = encoder.fit_transform(train[high_card_cols], train["is_late"])
test[high_card_cols] = encoder.transform(test[high_card_cols])

In [32]:
print(TargetEncoder.__module__)

sklearn.preprocessing._target_encoder


In [34]:
train = pd.get_dummies(train, columns=["payment_type"], prefix="pay", dummy_na=True)
test = pd.get_dummies(test, columns=["payment_type"], prefix="pay", dummy_na=True)
test = test.reindex(columns=train.columns, fill_value=0)

In [36]:
train

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,was_delivered,order_item_id,...,item_count,product_volume_cm3,same_state,seller_late_rate_hist,category_avg_freight_hist,pay_boleto,pay_credit_card,pay_debit_card,pay_voucher,pay_nan
35618,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,True,3,...,3,4096.0,0,0.222222,18.489299,False,False,False,False,True
35616,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,True,1,...,3,4096.0,0,0.222222,18.489299,False,False,False,False,True
35617,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,True,2,...,3,4096.0,0,0.222222,18.489299,False,False,False,False,True
107836,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,True,1,...,1,4096.0,0,0.104651,16.311272,True,False,False,False,False
32926,be5bc2f0da14d8071e2d45451ad119d9,7ec40b22510fdbea1b08921dd39e63d8,delivered,2016-10-03 16:56:50,2016-10-06 16:03:44,2016-10-21 16:33:46,2016-10-27 18:19:38,2016-11-07,True,1,...,1,4096.0,0,0.000000,19.150044,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12904,94fd9673a19164ebf42ba85826aff8b4,bd5a96b05e3979db414986300a0ca0a6,delivered,2018-05-24 10:43:56,2018-05-24 10:59:32,2018-05-24 14:24:00,2018-05-25 20:17:42,2018-06-11,True,1,...,1,7500.0,1,0.200000,18.712410,False,True,False,False,False
109507,d3085edb0a0b0d550bd72f712de90c09,1ebe4440e27e63e562784cd1b8a85a37,delivered,2018-05-24 11:00:54,2018-05-24 14:56:51,2018-05-25 12:44:00,2018-06-04 15:48:26,2018-06-19,True,1,...,2,24000.0,0,0.072000,19.749519,False,True,False,False,False
109506,d3085edb0a0b0d550bd72f712de90c09,1ebe4440e27e63e562784cd1b8a85a37,delivered,2018-05-24 11:00:54,2018-05-24 14:56:51,2018-05-25 12:44:00,2018-06-04 15:48:26,2018-06-19,True,1,...,2,24000.0,0,0.072000,19.749519,False,True,False,False,False
32813,bb05bd3bbacf1e3c6026b43b44a6631c,d6df67c855b6f445879822f6346dc45d,delivered,2018-05-24 11:05:40,2018-05-25 02:55:03,2018-05-30 12:58:00,2018-06-01 22:50:46,2018-06-11,True,1,...,1,7350.0,1,0.000000,19.150044,True,False,False,False,False


In [37]:
test

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,was_delivered,order_item_id,...,item_count,product_volume_cm3,same_state,seller_late_rate_hist,category_avg_freight_hist,pay_boleto,pay_credit_card,pay_debit_card,pay_voucher,pay_nan
55463,9ba2e8784e3e43f617fd0faa356d309d,475cd64aa7fcd579a93e4db98984450c,delivered,2018-05-24 11:07:21,2018-05-26 02:31:54,2018-05-26 08:16:00,2018-05-28 17:27:48,2018-06-11,True,1,...,1,600.0,1,0.072516,17.712968,True,False,False,False,False
40338,42a19395ca15b0662870b6b41be02edc,71e5e75dfaa74b62bfb191e28152d528,delivered,2018-05-24 11:11:42,2018-05-24 11:39:10,2018-05-28 13:20:00,2018-06-01 16:07:22,2018-06-29,True,1,...,1,11475.0,0,0.197279,18.085077,False,True,False,False,False
76503,47a1cdc1ae52b80a3800fb4509e657e4,8117d08aa8872ba4d8660d2b6c33c72a,delivered,2018-05-24 11:19:07,2018-05-24 11:39:23,2018-05-25 14:20:00,2018-06-04 21:42:29,2018-06-25,True,1,...,1,7140.0,0,0.111437,18.489299,False,True,False,False,False
23486,e148d34dad8db4dc18a911c2a74413d7,5ddcd01413e76d53e9ee604f43d65430,delivered,2018-05-24 11:28:18,2018-05-24 11:53:22,2018-05-25 13:16:00,2018-06-07 23:52:45,2018-07-03,True,1,...,1,6006.0,0,0.133333,21.840759,False,True,False,False,False
110367,4b26d2fce6fdc2f2a9f87c7cb05226ef,ba24be7f82de71a9ad1d254ff5198815,delivered,2018-05-24 11:29:05,2018-05-26 02:18:34,2018-05-28 12:24:00,2018-06-09 15:58:42,2018-06-25,True,1,...,1,4680.0,0,0.000000,23.611206,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34841,0b223d92c27432930dfe407c6aea3041,e60df9449653a95af4549bbfcb18a6eb,delivered,2018-08-29 14:18:23,2018-08-29 14:31:07,2018-08-29 15:29:00,2018-08-30 16:24:55,2018-09-04,True,1,...,2,115248.0,1,0.133333,44.822163,False,True,False,False,False
34842,0b223d92c27432930dfe407c6aea3041,e60df9449653a95af4549bbfcb18a6eb,delivered,2018-08-29 14:18:23,2018-08-29 14:31:07,2018-08-29 15:29:00,2018-08-30 16:24:55,2018-09-04,True,2,...,2,115248.0,1,0.133333,44.822163,False,True,False,False,False
36520,168626408cb32af0ffaf76711caae1dc,6e353700bc7bcdf6ebc15d6de16d7002,delivered,2018-08-29 14:18:28,2018-08-29 14:30:23,2018-08-29 18:51:00,2018-08-30 16:52:31,2018-09-11,True,1,...,1,13888.0,1,0.029703,18.489299,False,False,False,True,False
80406,03ef5dedbe7492bdae72eec50764c43f,496630b6740bcca28fce9ba50d8a26ef,delivered,2018-08-29 14:52:00,2018-08-29 15:05:22,2018-08-29 20:01:00,2018-08-30 16:36:59,2018-09-03,True,1,...,1,4788.0,1,0.085374,19.481429,False,True,False,False,False


# Phase 5 - Scaling

In [38]:
numeric_cols = ["price", "freight_value", "freight_to_price_ratio",
                 "product_weight_g", "product_volume_cm3",
                 "seller_late_rate_hist", "category_avg_freight_hist"]

In [39]:
scaler = StandardScaler()
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])
test[numeric_cols] = scaler.transform(test[numeric_cols])

In [40]:
feature_cols = numeric_cols + ["item_count", "same_state", "is_weekend",
                                "purchase_weekday", "purchase_month", "purchase_hour"]

# Phase 6 - Feature Selection

In [41]:
corr = train[feature_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.9)]
feature_cols = [c for c in feature_cols if c not in to_drop_corr]

In [42]:
X_train = train[feature_cols].fillna(0)
y_train = train["is_late"]

In [43]:
lasso = LogisticRegression(penalty="l1", solver="liblinear", C=0.1, random_state=RANDOM_STATE)
lasso.fit(X_train, y_train)

,penalty,'l1'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,123
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [44]:
lasso_importance = pd.Series(np.abs(lasso.coef_[0]), index=feature_cols).sort_values(ascending=False)
zeroed = lasso_importance[lasso_importance == 0].index.tolist()

In [45]:
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

In [47]:
rf_importance

seller_late_rate_hist        0.141047
freight_to_price_ratio       0.115561
freight_value                0.112350
purchase_hour                0.111447
price                        0.096480
product_volume_cm3           0.091854
product_weight_g             0.090330
purchase_weekday             0.068449
purchase_month               0.066183
category_avg_freight_hist    0.058008
item_count                   0.025992
is_weekend                   0.013501
same_state                   0.008798
dtype: float64

In [48]:
weak_in_both = [f for f in feature_cols if f in zeroed and rf_importance[f] < rf_importance.median()]
final_features = [c for c in feature_cols if c not in weak_in_both]

In [49]:
print(pd.DataFrame({"lasso": lasso_importance, "rf": rf_importance}))
print(f"\nDropped by correlation: {to_drop_corr}")
print(f"Dropped by model agreement: {weak_in_both}")

                              lasso        rf
category_avg_freight_hist  0.036215  0.058008
freight_to_price_ratio     0.044474  0.115561
freight_value              0.074856  0.112350
is_weekend                 0.019225  0.013501
item_count                 0.098511  0.025992
price                      0.000123  0.096480
product_volume_cm3         0.000000  0.091854
product_weight_g           0.004095  0.090330
purchase_hour              0.007425  0.111447
purchase_month             0.022114  0.066183
purchase_weekday           0.022293  0.068449
same_state                 0.718113  0.008798
seller_late_rate_hist      0.622388  0.141047

Dropped by correlation: []
Dropped by model agreement: []


# Phase 7 - Wrap-Up

In [50]:
summary = pd.DataFrame({
    "stage": ["Raw merged columns", "After feature creation", "After correlation filter", "After model-based check"],
    "feature_count": [raw_col_count, created_feature_count, len(feature_cols) + len(weak_in_both), len(final_features)]
})
print(summary)

print("\nLeakage boundary: all features derived only from data available at "
      "order_purchase_timestamp; seller/category history rates fit on train "
      "split only and mapped onto test.")

                      stage  feature_count
0        Raw merged columns             35
1    After feature creation             11
2  After correlation filter             13
3   After model-based check             13

Leakage boundary: all features derived only from data available at order_purchase_timestamp; seller/category history rates fit on train split only and mapped onto test.
